# Synthetic MAID Dataset

This notebook explains the synthetic dataset that powers the main demo path.

The updated synthetic generator models a neutral ad-tech retrieval problem:

- MAID-style user profiles
- publisher-scoped identity tokens
- explicit ad filters for card tier, geo, device, pacing, and frequency
- taxonomy-like scores plus derived retrieval segments


In [1]:
import json
from pathlib import Path

base = Path('../data/generated/synthetic')
metadata = json.loads((base / 'metadata.json').read_text())
metadata


{'num_users': 4000,
 'num_campaigns': 2500,
 'num_interactions': 120000,
 'feature_count': 12,
 'seed': 17,
 'wildcard_country_campaigns': 487,
 'wildcard_device_os_campaigns': 935,
 'wildcard_device_type_campaigns': 424,
 'wildcard_card_tier_campaigns': 556,
 'any_of_campaigns': 1796,
 'none_of_campaigns': 821,
 'state_targeted_campaigns': 1124,
 'postal_targeted_campaigns': 471,
 'paused_campaigns': 329}

## Dataset Files

The synthetic generator now writes:

- `maids.jsonl`: synthetic MAID profiles
- `identity_map.jsonl`: identity token to MAID mapping
- `campaigns.jsonl`: ad records with explicit targeting and delivery fields
- `interactions.parquet`: offline labels for evaluation
- `metadata.json`: generator summary


In [2]:
maids = [json.loads(line) for line in (base / 'maids.jsonl').read_text().splitlines()[:2]]
identity_rows = [json.loads(line) for line in (base / 'identity_map.jsonl').read_text().splitlines()[:5]]
campaigns = [json.loads(line) for line in (base / 'campaigns.jsonl').read_text().splitlines()[:2]]

maids[0], identity_rows[:3], campaigns[0]


({'age_bucket': '35-44',
  'card_tier': 'Standard',
  'device': 'Android',
  'device_type': 'tablet',
  'frequency_history': {'c00015': 2,
   'c00141': 2,
   'c00202': 1,
   'c00378': 1,
   'c00731': 2,
   'c00908': 1,
   'c00918': 1,
   'c01193': 1,
   'c01258': 1,
   'c01371': 3,
   'c01873': 2,
   'c02484': 2},
  'geo': 'CA',
  'identity_tokens': ['id_00000_01', 'id_00000_02', 'id_00000_03'],
  'impression_count': 25,
  'interests': {'camping': 0.3322,
   'family': 0.1344,
   'finance': 0.4761,
   'fitness': 0.4114,
   'foodie': 0.1935,
   'gaming': 0.6023,
   'home_improvement': 0.875,
   'luxury': 0.1125,
   'pet_care': 0.2847,
   'streaming': 0.4398,
   'tech': 0.4962,
   'travel': 0.7217},
  'postal_code': 'M4B1B3',
  'segments': ['home_improvement_high', 'travel_high', 'gaming_medium'],
  'spend_tier': 'low',
  'state': 'ON',
  'user_id': 'maid_00000'},
 [{'identity_token': 'id_00000_01', 'user_id': 'maid_00000'},
  {'identity_token': 'id_00000_02', 'user_id': 'maid_00000'},
  

## What A MAID Profile Contains

Each profile includes:

- `user_id` used as the synthetic MAID
- `identity_tokens`
- `geo`, `state`, and `postal_code`
- `device_type` and device OS
- `card_tier` and `spend_tier`
- taxonomy-like interest scores
- retrieval `segments`
- per-campaign `frequency_history`

The `interests` map is the synthetic stand-in for the taxonomy score map described in the target use case.


In [3]:
maid = maids[0]
{
    'maid': maid['user_id'],
    'identity_tokens': maid['identity_tokens'],
    'geo': {k: maid[k] for k in ['geo', 'state', 'postal_code']},
    'device': {k: maid[k] for k in ['device_type', 'device']},
    'carding': {k: maid[k] for k in ['card_tier', 'spend_tier']},
    'sample_segments': maid['segments'][:5],
}


{'maid': 'maid_00000',
 'identity_tokens': ['id_00000_01', 'id_00000_02', 'id_00000_03'],
 'geo': {'geo': 'CA', 'state': 'ON', 'postal_code': 'M4B1B3'},
 'device': {'device_type': 'tablet', 'device': 'Android'},
 'carding': {'card_tier': 'Standard', 'spend_tier': 'low'},
 'sample_segments': ['home_improvement_high', 'travel_high', 'gaming_medium']}

## What A Campaign Contains

Campaigns now have more explicit filtering fields than the original version:

- `card_tiers`
- `geo` for countries
- `geo_states`
- `geo_postal_codes`
- `device_types`
- `device` for OS targeting
- `pacing_status`
- `daily_budget_usd`
- `spent_today_usd`
- `frequency_cap`
- segment targeting fields

The reranker still uses `weights`, `bid`, and `freshness_boost` after eligibility has been enforced.


In [4]:
campaign = campaigns[0]
{
    'campaign_id': campaign['campaign_id'],
    'card_tiers': campaign['card_tiers'],
    'geo': {k: campaign[k] for k in ['geo', 'geo_states', 'geo_postal_codes']},
    'device': {k: campaign[k] for k in ['device_types', 'device']},
    'delivery': {k: campaign[k] for k in ['pacing_status', 'daily_budget_usd', 'spent_today_usd', 'frequency_cap']},
}


{'campaign_id': 'c00000',
 'card_tiers': ['Platinum', 'WorldElite'],
 'geo': {'geo': ['AU'], 'geo_states': [], 'geo_postal_codes': []},
 'device': {'device_types': ['tablet'], 'device': ['iOS']},
 'delivery': {'pacing_status': 'active',
  'daily_budget_usd': 7518.68,
  'spent_today_usd': 2422.13,
  'frequency_cap': 4}}

## Why The Dataset Uses Both Taxonomy Scores And Segments

The demo splits the logic intentionally:

- segments drive coarse Redis retrieval
- taxonomy-like interest scores drive the reranker

That mirrors a common serving pattern where Redis shrinks the candidate pool quickly and the app performs the exact scoring step.
